# CardPerks — Parser LoRA Training
Train a Qwen2.5-1.5B-Instruct LoRA to extract transactions from raw bank statement text.

Runtime: **T4 GPU** (~10 min)

In [ ]:
# Step 1 — Install deps (torchao first to avoid version conflict)
!pip install -q -U torchao
import os; os.kill(os.getpid(), 9)  # restart runtime after torchao upgrade

In [ ]:
!pip install -q -U transformers peft accelerate datasets huggingface-hub bitsandbytes

In [ ]:
# Step 2 — Upload parse_train.csv
from google.colab import files
import os
os.makedirs('data', exist_ok=True)
uploaded = files.upload()   # select cardperks-llm/data/parse_train.csv
import shutil
for fn in uploaded:
    shutil.move(fn, f'data/{fn}')
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Step 3 — Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Step 4 — Config
MODEL_ID   = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = 'qwen-cardperks-parser'
TRAIN_PATH = 'data/parse_train.csv'
MAX_LEN    = 160

PARSE_PROMPT = (
    'Extract transaction from bank statement entry. Output MERCHANT|AMOUNT or SKIP.\n'
    'Entry: {text}\n'
    'Result:'
)
print('Config ready')

In [ ]:
# Step 5 — Load and tokenize data
import csv
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

rows = []
with open(TRAIN_PATH, newline='') as f:
    for row in csv.DictReader(f):
        rows.append({'text': row['text'], 'label': row['label']})

print(f'Loaded {len(rows)} training rows')

# Quick sanity peek
import random
for r in random.sample(rows, 3):
    print(f"  [{r['label']}]  ←  {r['text'][:60].replace(chr(10),' ↵ ')}")

def tokenize(examples):
    inputs, labels_all = [], []
    for text, label in zip(examples['text'], examples['label']):
        prompt     = PARSE_PROMPT.format(text=text)
        full_text  = prompt + ' ' + label
        tok_full   = tokenizer(full_text,  truncation=True, max_length=MAX_LEN)
        tok_prompt = tokenizer(prompt,     truncation=True, max_length=MAX_LEN)
        input_ids  = tok_full['input_ids']
        prompt_len = len(tok_prompt['input_ids'])
        labels_all.append([-100] * prompt_len + input_ids[prompt_len:])
        inputs.append(input_ids)
    return {'input_ids': inputs, 'labels': labels_all}

dataset  = Dataset.from_list(rows)
tokenized = dataset.map(tokenize, batched=True, remove_columns=['text', 'label'])
print(f'Tokenized {len(tokenized)} examples')

In [ ]:
# Step 6 — Load model + apply LoRA
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map='auto',
)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Step 7 — Train
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=25,
    save_steps=200,
    save_total_limit=1,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, label_pad_token_id=-100),
)

trainer.train()

In [ ]:
# Step 8 — Save and download
import shutil
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}.zip')
files.download(f'{OUTPUT_DIR}.zip')

In [ ]:
# Step 9 — Sanity check (run a few examples through the trained model)
model.eval()

test_entries = [
    '05/27 Trader Joe S #041 San Luis Obispo CA Card 1449\n$33.89$4,500.59',
    '05/04Zelle Payment From Alex Johnson 9570053092$-240.84$4,221.52',
    'Beginning Balance$4,862.17',
    '03/15 STARBUCKS 800-782-7282 CA  6.75',
    'Chipotle Mexican Grill  06/12/2025  $14.50',
    '06/22 Netflix.com $15.99 $3,450.12',
]

for entry in test_entries:
    prompt = PARSE_PROMPT.format(text=entry)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    result = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    preview = entry.replace('\n', ' ↵ ')[:55]
    print(f'  [{result}]  ←  {preview}')